In [1]:
import pandas as pd

personality_data = pd.read_csv("data/Lab 3 - Personality Profile Type.csv")
personality_data.head()

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw|||...
1,ENTP,'I'm finding the lack of me in these posts ver...
2,INTP,'Good one _____ https://www.youtube.com/wat...
3,INTJ,"'Dear INTP, I enjoyed our conversation the o..."
4,ENTJ,'You're fired.|||That's another silly misconce...


In [2]:
personality_data["type"].value_counts()

type
INFP    1832
INFJ    1470
INTP    1304
INTJ    1091
ENTP     685
ENFP     675
ISTP     337
ISFP     271
ENTJ     231
ISTJ     205
ENFJ     190
ISFJ     166
ESTP      89
ESFP      48
ESFJ      42
ESTJ      39
Name: count, dtype: int64

In [3]:
personality_data["type"].nunique()

16

In [4]:
import random
random_indicies = random.sample(range(len(personality_data)), 10)
for i in random_indicies:
    print(f"index: {i}\nType: {personality_data['type'][i]}\nPosts: {personality_data['posts'][i]}\n\n")

index: 8283
Type: INFJ
Posts: 'Is he willing to use public restrooms? I ask, because if it gets to the point where he avoids using any other bathroom altogether (like, he will hold everything in for hours until he is safe in his...|||I'm talking about between the ages of about six and thirteen...  A Story:   When I was in 4th Grade, my teacher got up in front of the class one morning and told us about an article she had just...|||well, if it's completely empty, then I'd have to take the dirver's seat to go anywhere...;)  If it's a given that there's already a driver, then I'd probably sit about three or four rows back on...|||Your Se was probably taking over in the moment...|||To clarify. Yes, the order of the functions in socionics is as you say. However, according to Model A, they are not ordered by strength. Functions 1, 2, 7, and 8 are the stronger functions; and 3, 4,...|||When you posted your song a while back and talked about how you could play by ear and taught yourself...I had

In [5]:
personality_data.duplicated().sum()

np.int64(0)

In [6]:
personality_data.isnull().sum()

type     0
posts    0
dtype: int64

In [7]:
personality_data["type"].value_counts(normalize=True)*100

type
INFP    21.118156
INFJ    16.945245
INTP    15.031700
INTJ    12.576369
ENTP     7.896254
ENFP     7.780980
ISTP     3.884726
ISFP     3.123919
ENTJ     2.662824
ISTJ     2.363112
ENFJ     2.190202
ISFJ     1.913545
ESTP     1.025937
ESFP     0.553314
ESFJ     0.484150
ESTJ     0.449568
Name: proportion, dtype: float64

* IDK if this is a good approach but i see data rows isnt much yet the text that will be given to rnn is massive
* i realize that for each row its about posts and depend on all the posts in understand type of user
* i will split data on the ||| and for each of them new row with the same output will increase number of rows to train model on and in same time limit number of text entering the GRU model i will build later for memory loss controll

In [8]:
personality_data["posts"] = personality_data["posts"].apply(lambda x: str(x).split("|||"))
exploded_data = personality_data.explode("posts").reset_index(drop=True)
exploded_data.head()

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw
1,INFJ,http://41.media.tumblr.com/tumblr_lfouy03PMA1q...
2,INFJ,enfp and intj moments https://www.youtube.com...
3,INFJ,What has been the most life-changing experienc...
4,INFJ,http://www.youtube.com/watch?v=vXZeYwwRDw8 h...


In [9]:
exploded_data.shape

(422845, 2)

In [10]:
for row in exploded_data.itertuples():
	if row.posts is None or str(row.posts).strip() == "":
		print(f"index: {row.Index}\nType: {row.type}\nPosts: '{row.posts}'\n\n")

index: 2711
Type: INFP
Posts: ''


index: 2713
Type: INFP
Posts: ''


index: 2715
Type: INFP
Posts: ''


index: 2716
Type: INFP
Posts: ''


index: 2717
Type: INFP
Posts: ''


index: 8085
Type: INTJ
Posts: ''


index: 8086
Type: INTJ
Posts: ''


index: 8087
Type: INTJ
Posts: ''


index: 8088
Type: INTJ
Posts: ''


index: 8090
Type: INTJ
Posts: ''


index: 8091
Type: INTJ
Posts: ''


index: 8092
Type: INTJ
Posts: ''


index: 8094
Type: INTJ
Posts: ''


index: 8095
Type: INTJ
Posts: ''


index: 8096
Type: INTJ
Posts: ''


index: 9725
Type: ESFP
Posts: ''


index: 9726
Type: ESFP
Posts: ''


index: 10085
Type: INTP
Posts: ''


index: 10087
Type: INTP
Posts: ''


index: 10088
Type: INTP
Posts: ''


index: 10089
Type: INTP
Posts: ''


index: 14322
Type: ENTP
Posts: ''


index: 14323
Type: ENTP
Posts: ''


index: 14324
Type: ENTP
Posts: ''


index: 14325
Type: ENTP
Posts: ''


index: 14326
Type: ENTP
Posts: ''


index: 14328
Type: ENTP
Posts: ''


index: 14330
Type: ENTP
Posts: ''


index: 14

In [11]:
exploded_data = exploded_data.dropna(subset=["posts"])
exploded_data = exploded_data[exploded_data["posts"].str.strip() != ""]
exploded_data.head()

,type,posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw
1,INFJ,http://41.media.tumblr.com/tumblr_lfouy03PMA1q...
2,INFJ,enfp and intj moments https://www.youtube.com...
3,INFJ,What has been the most life-changing experienc...
4,INFJ,http://www.youtube.com/watch?v=vXZeYwwRDw8 h...


In [12]:
exploded_data.shape

(421757, 2)

In [13]:
exploded_data["type"].value_counts()

type
INFP    89595
INFJ    71948
INTP    63251
INTJ    52406
ENTP    33544
ENFP    32610
ISTP    16454
ISFP    12971
ENTJ    11235
ISTJ     9870
ENFJ     9283
ISFJ     8114
ESTP     4329
ESFP     2213
ESFJ     2018
ESTJ     1916
Name: count, dtype: int64

In [14]:
exploded_data["type"].value_counts(normalize=True)*100

type
INFP    21.243275
INFJ    17.059112
INTP    14.997024
INTJ    12.425638
ENTP     7.953395
ENFP     7.731940
ISTP     3.901299
ISFP     3.075468
ENTJ     2.663856
ISTJ     2.340210
ENFJ     2.201030
ISFJ     1.923857
ESTP     1.026420
ESFP     0.524710
ESFJ     0.478475
ESTJ     0.454290
Name: proportion, dtype: float64

* i increased number of data points for all which 
* will this lead to better learning for small classes after increasing its samples?

In [15]:
from sklearn.base import BaseEstimator, TransformerMixin
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re
import contractions
from nltk.tokenize import word_tokenize

class Clean_Text(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.lemmetizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words("english"))
        
    def fit(self, X, y=None):
        return self
    
    def transform(self, X: pd.Series):
        X =  X.copy()
        X_transformed = X.apply(self.Cleaner)
        return X_transformed
    
    def Cleaner(self, text):
        text = re.sub(r'http[s]?://\S+', '', str(text)).lower()
        text = contractions.fix(text)
        text = re.sub(r'[^a-z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        tokens = word_tokenize(text)
        tokens = [self.lemmetizer.lemmatize(word) for word in tokens if word not in self.stop_words]
        return ' '.join(tokens)
        

In [16]:
from sklearn.pipeline import Pipeline
test_pipeline = Pipeline([
    ("clean_text", Clean_Text())
])
cleaned_post_test = test_pipeline.fit_transform(exploded_data["posts"])
cleaned_post_test.shape

(421757,)

In [17]:
cleaned_post_test.head()

0                                                    
1                                                    
2    enfp intj moment sportscenter top ten play prank
3                       life changing experience life
4                                        repeat today
Name: posts, dtype: object

In [18]:
empty_data = cleaned_post_test[cleaned_post_test.str.strip() == ""]
empty_data.shape

(13887,)

In [19]:
exploded_data["Cleaned_Posts"] = cleaned_post_test
exploded_data.head()

,type,posts,Cleaned_Posts
0,INFJ,'http://www.youtube.com/watch?v=qsXHcwe3krw,
1,INFJ,http://41.media.tumblr.com/tumblr_lfouy03PMA1q...,
2,INFJ,enfp and intj moments https://www.youtube.com...,enfp intj moment sportscenter top ten play prank
3,INFJ,What has been the most life-changing experienc...,life changing experience life
4,INFJ,http://www.youtube.com/watch?v=vXZeYwwRDw8 h...,repeat today


In [20]:
final_cleaned_data = exploded_data[exploded_data["Cleaned_Posts"].str.strip() != ""].reset_index(drop=True)
final_cleaned_data.head()

,type,posts,Cleaned_Posts
0,INFJ,enfp and intj moments https://www.youtube.com...,enfp intj moment sportscenter top ten play prank
1,INFJ,What has been the most life-changing experienc...,life changing experience life
2,INFJ,http://www.youtube.com/watch?v=vXZeYwwRDw8 h...,repeat today
3,INFJ,May the PerC Experience immerse you.,may perc experience immerse
4,INFJ,The last thing my INFJ friend posted on his fa...,last thing infj friend posted facebook committ...


In [21]:
final_cleaned_data.isnull().sum()

type             0
posts            0
Cleaned_Posts    0
dtype: int64

In [22]:
empty_rows = final_cleaned_data[final_cleaned_data["Cleaned_Posts"].str.strip() == ""]
print(empty_rows)

Empty DataFrame
Columns: [type, posts, Cleaned_Posts]
Index: []


In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(final_cleaned_data["Cleaned_Posts"], final_cleaned_data["type"], test_size=0.2, random_state=42, stratify=final_cleaned_data["type"], shuffle=True)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=42, stratify=y_train, shuffle=True)
print(f"Train: {X_train.shape[0]} samples")
print(f"Validation: {X_valid.shape[0]} samples")
print(f"Test: {X_test.shape[0]} samples")

Train: 261036 samples
Validation: 65260 samples
Test: 81574 samples


In [24]:
import torch.nn as nn

class GRU_Model(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden_dim, num_classes, num_layers=1):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)
        self.RLU = nn.ReLU()
        
    def forward(self, x):
        embedd = self.embedding(x)
        gru_output, hidden = self.gru(embedd)
        output = self.RLU(gru_output[:, -1, :])
        output = self.fc(output)
        return output

In [25]:
from collections import Counter

all_text = " ".join(final_cleaned_data["Cleaned_Posts"])
word_counts = Counter(all_text.split())
frequent_words = [word for word, count in word_counts.items() if count >= 3]
print(f"Number of unique words: {len(word_counts)}")
print(f"Unique words after filtering: {len(frequent_words)}")

Number of unique words: 88766
Unique words after filtering: 35123


In [26]:
vocab = ["<PAD>", "<UNK>"] + frequent_words
vocab_size = len(vocab)
w2i = {word: idx for idx, word in enumerate(vocab)}
i2w = {idx: word for idx, word in enumerate(vocab)}
print(f"index for pad is:{w2i['<PAD>']}\nindex for unk is:{w2i['<UNK>']}\n")
print(f"index of 100 its word is:{i2w[99]}")

index for pad is:0
index for unk is:1

index of 100 its word is:late


In [27]:
import torch
from sklearn.base import BaseEstimator, TransformerMixin
class TextToVector(BaseEstimator, TransformerMixin):
    def __init__(self, w2i, max_len):
        self.w2i = w2i
        self.max_len = max_len
    
    def fit(self, X, y=None):
        return self
    
    def encode_pad_seq(self, seq):
        encoded = [self.w2i.get(word, self.w2i["<UNK>"]) for word in str(seq).split()]
        
        if len(encoded) > self.max_len:
            return encoded[:self.max_len]
         
        elif len(encoded) < self.max_len:
            return encoded + [self.w2i["<PAD>"]] * (self.max_len - len(encoded))
        else:
            return encoded
    
    def transform(self, X):
        encoded = X.apply(self.encode_pad_seq)
        encoded_list = encoded.tolist()
        return torch.tensor(encoded_list, dtype=torch.long)

In [28]:
final_pipeline = Pipeline([
    ("cleaner", Clean_Text()),
    ("vectorizer", TextToVector(w2i, max_len=100))
])
final_pipeline.fit(X_train)
X_train_vectorized = final_pipeline.transform(X_train)
X_train_vectorized.shape

d:\anaconda\envs\iti-nlp\lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


torch.Size([261036, 100])

In [29]:
X_valid_vectorized = final_pipeline.transform(X_valid)
X_valid_vectorized.shape

d:\anaconda\envs\iti-nlp\lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


torch.Size([65260, 100])

In [30]:
X_test_vectorized = final_pipeline.transform(X_test)
X_test_vectorized.shape

d:\anaconda\envs\iti-nlp\lib\site-packages\sklearn\pipeline.py:61: FutureWarning: This Pipeline instance is not fitted yet. Call 'fit' with appropriate arguments before using other methods such as transform, predict, etc. This will raise an error in 1.8 instead of the current warning.
  warnings.warn(


torch.Size([81574, 100])

In [32]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_valid_encoded = label_encoder.transform(y_valid)
y_test_encoded = label_encoder.transform(y_test)
print(f"Encoded labels: {label_encoder.classes_}")
print(f"length is: {len(label_encoder.classes_)}")

Encoded labels: ['ENFJ' 'ENFP' 'ENTJ' 'ENTP' 'ESFJ' 'ESFP' 'ESTJ' 'ESTP' 'INFJ' 'INFP'
 'INTJ' 'INTP' 'ISFJ' 'ISFP' 'ISTJ' 'ISTP']
length is: 16


In [33]:
y_train_encoded = torch.tensor(y_train_encoded, dtype=torch.long)
y_valid_encoded = torch.tensor(y_valid_encoded, dtype=torch.long)
y_test_encoded = torch.tensor(y_test_encoded, dtype=torch.long)

In [34]:
from torch.utils.data import TensorDataset, DataLoader
train_dataset = TensorDataset(X_train_vectorized, y_train_encoded)
valid_dataset = TensorDataset(X_valid_vectorized, y_valid_encoded)
test_dataset = TensorDataset(X_test_vectorized, y_test_encoded)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [35]:
import torch.optim as optim
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [36]:
criterion = nn.CrossEntropyLoss()
model = GRU_Model(vocab_size=vocab_size, emb_dim=100, hidden_dim=128, num_classes=len(label_encoder.classes_))
optimizer = optim.Adam(model.parameters(), lr=0.001)

for i in range(10):
    model.train()
    total_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(train_loader)
    
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for batch_x, batch_y in valid_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            val_loss += loss.item()
    print(f"Epoch {i+1}, Loss: {avg_loss:.4f}, Val Loss: {val_loss:.4f}")

Epoch 1, Loss: 2.2770, Val Loss: 2285.7567
Epoch 2, Loss: 2.1984, Val Loss: 2219.0073
Epoch 3, Loss: 2.1397, Val Loss: 2216.3681


KeyboardInterrupt: 